In this script, the neighbours are determined for each petrol station based on the original data set using the travel time.

# 1. Import Data

In [1]:
# Standard libs
from __future__ import annotations
from dataclasses import dataclass
import os
import glob
import json

# Utils
import math
import random
import gc

# Data & Processing
import numpy as np
import pandas as pd
import geopandas as gpd

# Scikit-Learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, confusion_matrix
from sklearn.neighbors import BallTree, NearestNeighbors
from sklearn.model_selection import train_test_split

# TQDM
from tqdm.notebook import tqdm
from joblib import Parallel, delayed

# Plotting
import matplotlib.pyplot as plt

# PyTorch
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch_geometric.utils import subgraph

# PyTorch Geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv


# Confirm the current working directory
print("Current working directory:", os.getcwd())

Current working directory: /home/qusta100/STGNN/02_Notebooks/01_Preprocessing


In [3]:
#Load main data set
df = pd.read_csv(
    "/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/02_Analysis/final_df.csv",
    dtype={9: str}
)

# 2. Prepare Data

In [4]:
cols = ["station_uuid", "latitude", "longitude", "in_thuringia"]
stations = (
    df[cols]
    .dropna(subset=["latitude", "longitude", "in_thuringia"])
    .drop_duplicates(subset=["station_uuid"])
    .reset_index(drop=True)
)
stations.shape

(1033, 4)

In [5]:
# Create NumPy-Array
lat_lon_deg = stations[["latitude", "longitude"]].to_numpy(dtype=np.float64)  # (lat, lon)

The maximum limit of 22 km is used to reduce the number of potential neighbours in a first step, as the subsequent step – calculating the travel times between two petrol stations – is more computationally complex. The distance of 22 km corresponds to the maximum distance that can be covered within 10 minutes (assumption: 130 km/h, maximum speed in Germany).

In [6]:
def km_to_radians(km: float) -> float:
    return km / 6371.0088

def build_candidates_radius(lat_lon_deg: np.ndarray, radius_km: float = 22.0, max_candidates: int = 30):
    lat_lon_rad = np.radians(lat_lon_deg)
    tree = BallTree(lat_lon_rad, metric="haversine")

    r = km_to_radians(radius_km)
    ind, dist = tree.query_radius(lat_lon_rad, r=r, return_distance=True, sort_results=True)

    candidates, cand_dists_km = [], []
    for i in range(lat_lon_deg.shape[0]):
        neigh, d = ind[i], dist[i]

        # self raus
        if len(neigh) > 0 and neigh[0] == i:
            neigh, d = neigh[1:], d[1:]

        # cap
        if max_candidates is not None and len(neigh) > max_candidates:
            neigh, d = neigh[:max_candidates], d[:max_candidates]

        candidates.append(neigh.astype(np.int32))
        cand_dists_km.append((d * 6371.0088).astype(np.float32))

    return candidates, cand_dists_km

candidates, cand_dists_km = build_candidates_radius(lat_lon_deg, radius_km=22.0, max_candidates=30)

In [7]:
def candidates_to_edgelist_uuid(stations_df, lat_lon_deg, candidates, cand_dists_km):
    src_idx_chunks, dst_idx_chunks, dist_chunks = [], [], []

    for src, (dsts, dists) in enumerate(zip(candidates, cand_dists_km)):
        if len(dsts) == 0:
            continue
        src_idx_chunks.append(np.full(len(dsts), src, dtype=np.int32))
        dst_idx_chunks.append(dsts.astype(np.int32))
        dist_chunks.append(dists.astype(np.float32))

    src_idx = np.concatenate(src_idx_chunks) if src_idx_chunks else np.array([], dtype=np.int32)
    dst_idx = np.concatenate(dst_idx_chunks) if dst_idx_chunks else np.array([], dtype=np.int32)
    hav_km = np.concatenate(dist_chunks) if dist_chunks else np.array([], dtype=np.float32)

    src_uuid = stations_df.loc[src_idx, "station_uuid"].to_numpy()
    dst_uuid = stations_df.loc[dst_idx, "station_uuid"].to_numpy()

    return pd.DataFrame({
        "src_uuid": src_uuid,
        "dst_uuid": dst_uuid,
        "haversine_km": hav_km,
        "src_lat": lat_lon_deg[src_idx, 0].astype(np.float32),
        "src_lon": lat_lon_deg[src_idx, 1].astype(np.float32),
        "dst_lat": lat_lon_deg[dst_idx, 0].astype(np.float32),
        "dst_lon": lat_lon_deg[dst_idx, 1].astype(np.float32),
    })

edges = candidates_to_edgelist_uuid(stations, lat_lon_deg, candidates, cand_dists_km)
edges.shape, edges.head(10)

((30098, 7),
                                src_uuid                              dst_uuid  \
 0  00060065-7890-4444-8888-acdc00000004  e586f8db-b60a-44e5-93eb-63790f76c829   
 1  00060065-7890-4444-8888-acdc00000004  03aff575-0ff9-4816-9478-43f85469fdc6   
 2  00060065-7890-4444-8888-acdc00000004  5525b741-079a-4705-8aef-4948b32b77c6   
 3  00060065-7890-4444-8888-acdc00000004  e41f2c48-d414-4e95-ab8e-60fdd828dac5   
 4  00060065-7890-4444-8888-acdc00000004  ddb01c83-ad26-4e8e-87a8-870acfc599cd   
 5  00060065-7890-4444-8888-acdc00000004  187eaf2a-dc22-4daa-9af6-456f717fb62e   
 6  00060065-7890-4444-8888-acdc00000004  c55a0021-dd36-49d8-8056-8b9b535b35c8   
 7  00060065-7890-4444-8888-acdc00000004  51d4b65a-a095-1aa0-e100-80009459e03a   
 8  00060065-7890-4444-8888-acdc00000004  0723a4c7-583f-4d45-8c22-caa2d5e280d9   
 9  00060065-7890-4444-8888-acdc00000004  00060807-0001-4444-8888-acdc00000001   
 
    haversine_km    src_lat    src_lon    dst_lat    dst_lon  
 0      1.062471  50

In [9]:
out_path = "/home/qusta100/STGNN/01_Data/02_Processed_Data/01_OSMR/stations_candidates_r14_max30.csv.gz"
edges.to_csv(out_path, index=False, compression="gzip")
out_path

'/home/qusta100/STGNN/01_Data/02_Processed_Data/01_OSMR/stations_candidates_r14_max30.csv.gz'

# 3. Import Travel Times via OSRM

Travel times are calculated on a separate (local) server. For this purpose, a local OSRM routing server was set up, which works with current (non-dynamic) map data from Germany and can determine the travel time between two coordinates. Travel times are calculated in route_candidates.py. The file is then imported again.

In [3]:
tt = pd.read_csv("/home/qusta100/STGNN/01_Data/02_Processed_Data/01_OSMR/stations_travel_times.csv.gz")
tt["travel_time_min"].describe()

count    28985.000000
mean        16.706708
std          7.715090
min          0.013333
25%         10.861667
50%         16.803333
75%         22.185000
max         43.615000
Name: travel_time_min, dtype: float64

In [4]:
# Test for missing travel times (expected: 0)
tt.isna().sum()

src_uuid           0
dst_uuid           0
travel_time_min    0
dtype: int64

In [6]:
tt.head(10)

,src_uuid,dst_uuid,travel_time_min
0,00060065-7890-4444-8888-acdc00000004,e586f8db-b60a-44e5-93eb-63790f76c829,1.583333
1,00060065-7890-4444-8888-acdc00000004,03aff575-0ff9-4816-9478-43f85469fdc6,2.058333
2,00060065-7890-4444-8888-acdc00000004,5525b741-079a-4705-8aef-4948b32b77c6,2.758333
3,00060065-7890-4444-8888-acdc00000004,e41f2c48-d414-4e95-ab8e-60fdd828dac5,4.403333
4,00060065-7890-4444-8888-acdc00000004,ddb01c83-ad26-4e8e-87a8-870acfc599cd,5.015000
5,00060065-7890-4444-8888-acdc00000004,187eaf2a-dc22-4daa-9af6-456f717fb62e,3.053333
6,00060065-7890-4444-8888-acdc00000004,c55a0021-dd36-49d8-8056-8b9b535b35c8,8.198334
7,00060065-7890-4444-8888-acdc00000004,51d4b65a-a095-1aa0-e100-80009459e03a,6.280000
8,00060065-7890-4444-8888-acdc00000004,0723a4c7-583f-4d45-8c22-caa2d5e280d9,6.355000
9,00060065-7890-4444-8888-acdc00000004,00060807-0001-4444-8888-acdc00000001,11.101666
